# 캘리포니아 주택 가격 데이터셋

- [California Housing](https://github.com/ageron/data/tree/main/housing)
- 1990년 캘리포니아 인구 조사 데이터 기반

### 데이터 가져오기

- 데이터 다운로드
  - [HandsOn Github Repo.](https://github.com/ageron/handson-ml3)

In [4]:
from pathlib import Path
import pandas as pd
import tarfile
import urllib.request

def load_housing_data():
    tarball_path = Path("datasets/housing.tgz")
    if not tarball_path.is_file():
        Path("datasets").mkdir(parents=True, exist_ok=True)
        url = "https://github.com/ageron/data/raw/main/housing.tgz"
        urllib.request.urlretrieve(url, tarball_path)
    with tarfile.open(tarball_path) as housing_tarball:
            housing_tarball.extractall(path="datasets")
    return pd.read_csv(Path("datasets/housing/housing.csv"))

housing = load_housing_data()

### 데이터 구조 훑어보기

- head() 메서드:
  - 각 행은 하나의 구역을 표시함
  - 특성은 10개
- info() 메서드:
  - 전체 행 수
  - 각 특성 데이터 타입
  - NULL이 아닌 값의 개수 확인 

In [5]:
housing.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


In [6]:
housing.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   longitude           20640 non-null  float64
 1   latitude            20640 non-null  float64
 2   housing_median_age  20640 non-null  float64
 3   total_rooms         20640 non-null  float64
 4   total_bedrooms      20433 non-null  float64
 5   population          20640 non-null  float64
 6   households          20640 non-null  float64
 7   median_income       20640 non-null  float64
 8   median_house_value  20640 non-null  float64
 9   ocean_proximity     20640 non-null  object 
dtypes: float64(9), object(1)
memory usage: 1.6+ MB


- `ocean_proximity` 필드를 제외하면, 모든 특성이 숫자형이다. 
- value_counts() 메서드:
  - 범주형(categorical) 타입으로, 어떤 카테고리가 있는지 확인

In [7]:
housing["ocean_proximity"].value_counts()

ocean_proximity
<1H OCEAN     9136
INLAND        6551
NEAR OCEAN    2658
NEAR BAY      2290
ISLAND           5
Name: count, dtype: int64

- describe() 메서드
  - 숫자형 특성의 요약 정보 확인
  - count, mean, min, max
  - std: 표준 편차
  - 25%, 50%, 75%: 백분위수

In [8]:
housing.describe()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value
count,20640.000000,20640.000000,20640.000000,20640.000000,20433.000000,20640.000000,20640.000000,20640.000000,20640.000000
mean,-119.569704,35.631861,28.639486,2635.763081,537.870553,1425.476744,499.539680,3.870671,206855.816909
std,2.003532,2.135952,12.585558,2181.615252,421.385070,1132.462122,382.329753,1.899822,115395.615874
min,-124.350000,32.540000,1.000000,2.000000,1.000000,3.000000,1.000000,0.499900,14999.000000
25%,-121.800000,33.930000,18.000000,1447.750000,296.000000,787.000000,280.000000,2.563400,119600.000000
50%,-118.490000,34.260000,29.000000,2127.000000,435.000000,1166.000000,409.000000,3.534800,179700.000000
75%,-118.010000,37.710000,37.000000,3148.000000,647.000000,1725.000000,605.000000,4.743250,264725.000000
max,-114.310000,41.950000,52.000000,39320.000000,6445.000000,35682.000000,6082.000000,15.000100,500001.000000


### 데이터셋 구성하기

- 훈련 데이터셋(train dataset)
- 검증 데이터셋(validate dataset)
- 평가 데이터셋(test dataset)

어떤 인공지능 모델을 사용할지 결정하려면 데이터의 특성을 잘 살펴 보아야 한다. 여기서 주의 할 것이 있다. 우라가 갖고 있는 전체 데이터를 가지고 훈련/검증/평가를 수행해 해야 하는데, 데이터를 살펴 보는 과정에 우리의 뇌가 편향될 수 있수도 있다. 나도 모르는 사이에 특정 데이터를 평가 데이터(test dataset)으로 사용하는 오류를 범할 수도 있다. 따라서 데이터의 자세히 살펴 보기 전에, 평가 데이터셋을 랜덤하게 샘플링하여 20% 정도(데이터셋이 매우 크다면 그보다 적게) 떼어놓는 것이 안전하다.

- 평가 데이터 만들기

In [ ]:
import numpy as np

def shuttle_split(data, test_ratio):
    np.random.seed(42)  # For reproducible results
    
    shuffled_indices = np.random.permutation(len(data))
    test_set_size = int(len(data) * test_ratio)
    test_indices = shuffled_indices[:test_set_size]
    train_indices = shuffled_indices[test_set_size:]
    
    return data.iloc[train_indices], data.iloc[test_indices]

In [ ]:
# Using the function to split the housing data
train_set, test_set = shuttle_split(housing, 0.2)

- 평가 데이터 만들기

In [ ]:
from zlib import crc32

def is_id_in_test_set(identifier, test_ratio):
    return crc32(np.int64(identifier)) & 0xffffffff < test_ratio * 2**32

def shuttle_split_by_id(data, test_ratio, id_column):
    test_set = data[data[id_column].apply(lambda id_: is_id_in_test_set(id_, test_ratio))]
    train_set = data[~data[id_column].apply(lambda id_: is_id_in_test_set(id_, test_ratio))]
    return train_set, test_set

In [ ]:
housing_widh_id = housing.reset_index()  # Add an 'index' column as ID
train_set, test_set = shuttle_split_by_id(housing_widh_id, 0.2, "index")

>   or

In [ ]:
housing_widh_id["id"] = housing["longitude"] * 1000 + housing["latitude"]
train_set, test_set = shuttle_split_by_id(housing_widh_id, 0.2, "id")

- 평가 데이터 만들기

In [ ]:
from sklearn.model_selection import train_test_split
train_set, test_set = train_test_split(housing, test_size=0.2, random_state=42)